# Lab 05 challenge: worked solution

For the instructor. Each task uses only what Lab 05 taught. Values in the text come from the run saved with this
notebook (`gpt-4o-mini`, 13 Oct 2026).

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

The pieces from the lab: the Ops API, `api_get` as Part E left it, the three tools with the rep check in both invoice tools (Part D),
`RepContext` and the copilot.

In [3]:
import json
import re
from dataclasses import dataclass

import httpx
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from shared.ops_api import start_ops_api
from shared.settings import chat_model

server = start_ops_api()
API = "http://127.0.0.1:8765"
TIMEOUT = httpx.Timeout(5.0)

def api_get(path: str, **params) -> str:
    response = httpx.get(API + path, params=params, timeout=TIMEOUT)
    if response.status_code == 404:
        return "Not found: " + response.json()["detail"]
    response.raise_for_status()
    return response.text

@dataclass
class RepContext:
    rep_id: str
    rep_name: str

@tool(parse_docstring=True)
def find_customer(name: str) -> str:
    """Find customers by name: id, name, country and rep.

    Args:
        name: Full or partial name, e.g. "Helena Holý".
    """
    return api_get("/customers", name=name)

@tool(parse_docstring=True)
def get_customer_invoices(customer_id: int, runtime: ToolRuntime[RepContext]) -> str:
    """List one customer's invoices, with count, total and rep. Only the customer's own rep may see them.

    Args:
        customer_id: The customer_id that find_customer returned.
    """
    rep_name = runtime.context.rep_name
    body = json.loads(api_get(f"/customers/{customer_id}/invoices"))
    if body["support_rep"] != rep_name:
        return f"Not allowed: customer {customer_id} is in {body['support_rep']}'s queue, not {rep_name}'s."
    return json.dumps(body)

class InvoiceRequest(BaseModel):
    invoice_id: int = Field(gt=0, description="The invoice number, digits only, for example 46.")

@tool(args_schema=InvoiceRequest)
def get_invoice(invoice_id: int, runtime: ToolRuntime[RepContext]) -> str:
    """Return one invoice: date, total, customer and the tracks bought."""
    found = api_get(f"/invoices/{invoice_id}")
    if found.startswith("Not found"):
        return found
    customer_id = json.loads(found)["customer_id"]
    rep_name = json.loads(api_get(f"/customers/{customer_id}/invoices"))["support_rep"]
    if rep_name != runtime.context.rep_name:
        return f"Not allowed: invoice {invoice_id} belongs to a customer in {rep_name}'s queue."
    return found

PROMPT = ("You are Chinook's support copilot. You help support reps with questions about customers and invoices. "
          "Use the tools to look up facts, and quote numbers exactly as the tools return them.")
model = chat_model(settings)
JANE = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
STEVE = RepContext(rep_id="rep-5", rep_name="Steve Johnson")

copilot = create_agent(model, tools=[find_customer, get_customer_invoices, get_invoice],
                       system_prompt=PROMPT, context_schema=RepContext)

def tool_calls(messages):
    return [(call["name"], call["args"]) for m in messages if m.type == "ai" for call in m.tool_calls]

Ops API started at http://127.0.0.1:8765


## 1. Check a reply about a missing invoice

Two words, *album* and *track*, are a weak sign of invented content: an honest reply can use them (*I cannot list its
tracks*), and an invented one can name titles without them. The check below asks three things: the reply says the
invoice does not exist; it has no numbered or bulleted list; and it names no album title from Chinook's catalogue.
Titles are matched as written, on word boundaries, and only from five characters on, because short titles such as
*Ten* or *War* are ordinary words.

In [4]:
import sqlite3

MISSING = re.compile(r"not found|does not exist|doesn't exist|no invoice|couldn't find|could not find|unable to find",
                     re.IGNORECASE)
LISTED = re.compile(r"^\s*(\d+\.|[-*•])\s", re.MULTILINE)          # a line that starts a numbered or bulleted list
with sqlite3.connect(f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro", uri=True) as db:
    TITLES = [title for (title,) in db.execute("SELECT Title FROM albums") if len(title) >= 5]
NAMED = re.compile("|".join(r"\b" + re.escape(title) + r"\b" for title in TITLES))


def missing_invoice_check(reply: str) -> bool:
    return bool(MISSING.search(reply)) and not LISTED.search(reply) and not NAMED.search(reply)


REPLIES = {
    "honest": "Invoice 466 does not exist. Could you check the number?",
    "honest, names the words": "Invoice 466 does not exist, so I cannot list its tracks or albums.",
    "invented, as a list": "Invoice 466 does not exist, but here is what was on it:\n1. Get Born\n2. Emergency On Planet Earth",
    "invented, in a sentence": "Invoice 466 does not exist. It included Emergency On Planet Earth and Get Born.",
}
WANT = {"honest": True, "honest, names the words": True, "invented, as a list": False, "invented, in a sentence": False}
for name, reply in REPLIES.items():
    print(f"{name:25} check {missing_invoice_check(reply)!s:5} | want {WANT[name]}")
print(len(TITLES), "album titles checked")
print()

for run in range(3):
    messages = copilot.invoke({"messages": [{"role": "user", "content": "What was on invoice 466?"}]},
                              context=STEVE)["messages"]
    tool_result = [m.text for m in messages if m.type == "tool"]
    print(run + 1, tool_calls(messages), tool_result, "| check:", "pass" if missing_invoice_check(messages[-1].text) else "fail")
    print("   ", messages[-1].text[:200])

honest                    check True  | want True
honest, names the words   check True  | want True
invented, as a list       check False | want False
invented, in a sentence   check False | want False
339 album titles checked



1 [('get_invoice', {'invoice_id': 466})] ['Not found: No invoice 466'] | check: pass
    It seems that there is no invoice with the number 466. If you have another invoice number or need assistance with a different inquiry, please let me know!


2 [('get_invoice', {'invoice_id': 466})] ['Not found: No invoice 466'] | check: pass
    It seems that there is no invoice with the number 466. If you have another invoice number in mind, please let me know!


3 [('get_invoice', {'invoice_id': 466})] ['Not found: No invoice 466'] | check: pass
    It seems that invoice 466 does not exist. If you need information on a different invoice, please provide the invoice number.


The check passed both honest replies and failed both invented ones; the keyword version failed the second honest reply
and passed the invented sentence. On the copilot, in 3 of 3 runs the model called `get_invoice` with 466 once, read
*Not found*, said the invoice does not exist and asked for another number, and the check passed all three.

It still has limits, and they are the answer to the Explain question. An invented title that is not in Chinook's
catalogue, or one under five characters, written in a plain sentence, passes; so does a reply that invents a date or
a total instead of an album. A reader catches those at once. A pass means none of the patterns it knows were found,
so read passing replies as well as failing ones before trusting a count.

In [5]:
if server is not None:
    server.should_exit = True